#  Downloading Libraries

In [3]:
! pip install nltk
! pip install vaderSentiment
! pip install pytrends
! pip install textblob
! pip install wordcloud
! pip install gensim
! pip install seaborn
! pip install TextBlob
! pip install joblib

# Importing Libraries

In [5]:
import nltk
import logging
import numpy as np
import pandas as pd
import seaborn as sns
from textblob import TextBlob
import matplotlib.pyplot as plt
from nltk.corpus import stopwords

import re
import string 
import joblib
import warnings
warnings.filterwarnings('ignore')

In [6]:
# Download NLTK resources (if not already downloaded)
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('wordnet', quiet=True)

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/bishanbhandari/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/bishanbhandari/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/bishanbhandari/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [88]:
# Importing Dataset
df=pd.read_csv('Dataset/User Review/reviews.csv')
df.head()

,listing_id,id,date,reviewer_id,reviewer_name,comments
0,2992450,15066586,2014-07-01,16827297,Kristen,Large apartment; nice kitchen and bathroom. Ke...
1,2992450,21810844,2014-10-24,22648856,Christopher,"This may be a little late, but just to say Ken..."
2,2992450,27434334,2015-03-04,45406,Altay,The apartment was very clean and convenient to...
3,2992450,28524578,2015-03-25,5485362,John,Kenneth was ready when I got there and arrange...
4,2992450,35913434,2015-06-23,15772025,Jennifer,We were pleased to see how 2nd Street and the ...


# Data Preprocessing 

In [9]:
df.columns

Index(['listing_id', 'id', 'date', 'reviewer_id', 'reviewer_name', 'comments'], dtype='object')

In [10]:
df.shape

(29046, 6)

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29046 entries, 0 to 29045
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   listing_id     29046 non-null  int64 
 1   id             29046 non-null  int64 
 2   date           29046 non-null  object
 3   reviewer_id    29046 non-null  int64 
 4   reviewer_name  29046 non-null  object
 5   comments       29038 non-null  object
dtypes: int64(3), object(3)
memory usage: 1.3+ MB


In [12]:
df.describe()

,listing_id,id,reviewer_id
count,2.904600e+04,2.904600e+04,2.904600e+04
mean,3.605845e+17,9.183168e+17,2.196129e+15
std,5.209583e+17,5.589472e+17,6.067939e+16
min,2.992450e+06,1.506659e+07,2.830000e+02
25%,2.872227e+07,5.311163e+17,6.231337e+07
50%,4.805235e+07,1.025446e+18,1.824196e+08
75%,8.501735e+17,1.408826e+18,3.941900e+08
max,1.681403e+18,1.709451e+18,1.700398e+18


In [13]:
df.isnull().sum()

listing_id       0
id               0
date             0
reviewer_id      0
reviewer_name    0
comments         8
dtype: int64

In [14]:
df.duplicated().sum()

0

In [15]:
# Removing the columns from dataset
df=df.drop(['id','reviewer_id','listing_id'],axis=1)
df.head()

,date,reviewer_name,comments
0,2014-07-01,Kristen,Large apartment; nice kitchen and bathroom. Ke...
1,2014-10-24,Christopher,"This may be a little late, but just to say Ken..."
2,2015-03-04,Altay,The apartment was very clean and convenient to...
3,2015-03-25,John,Kenneth was ready when I got there and arrange...
4,2015-06-23,Jennifer,We were pleased to see how 2nd Street and the ...


In [16]:
# Removing URLs,Mentions and hashtag
def remove_urls(text):
    if not isinstance(text,str):
        logging.error("Input must be a string.")
        raise TypeError("Input must be a string.")

    # Removing mentions and hashtag from the text
    pattern=r'\@\w+|\#\w+'
    cleaned_text=re.sub(pattern,'',text)

    return cleaned_text
        

In [30]:
# preload stop words to avoid reloading them every time the function is called 
stop_words=set(stopwords.words('english'))

def remove_stopwords(text):
    if not isinstance(text,str):
        logging.error("Input must be a string.")
        raise TypeError("Input must be a string.")

    # removing stop words from the text
    filtered_words=[word for word in text.split() if word not in stop_words]
    cleaned_text=' '.join(filtered_words)

    return cleaned_text

In [32]:
def remove_numbersr(text):
    """Remove numbers from a text string"""
    return re.sub(r'\d+','',text)

In [40]:
def remove_emojis(text):
    """Remove emojis from a text string"""
    #[] define character (match any one character class)
    emoji_pattern=re.compile(pattern="["
                             u"\U0001F600-\U0001F64F"  # emoticons
                               u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                               u"\U0001F680-\U0001F6FF"  # transport & map symbols
                               u"\U0001F700-\U0001F77F"  # alchemical symbols
                               u"\U0001F780-\U0001F7FF"  # Geometric Shapes Extended
                               u"\U0001F800-\U0001F8FF"  # Supplemental Arrows-C
                               u"\U0001F900-\U0001F9FF"  # Supplemental Symbols and Pictographs
                               u"\U0001FA00-\U0001FA6F"  # Chess Symbols
                               u"\U0001FA70-\U0001FAFF"  # Symbols and Pictographs Extended-A
                               u"\U00002702-\U000027B0"  # Dingbats
                               u"\U000024C2-\U0001F251"
                             "]+",flags=re.UNICODE)
    return emoji_pattern.sub(r'',text)

In [42]:
# replace whitespace
def handle_whitespace(text):
    return re.sub(r'\s+',' ',text).strip()

In [44]:
def lemmitization(text):
    lemmitizer=nltk.stem.WordNetLemmatizer()
    return ' '.join([lemmitizer.lemmatize(word) for word in text.split()])

In [48]:
def stemminization(text):
    stemmer=nltk.stem.PorterStemmer()
    return ' '.join([stemmer.stem(word) for word in text.split()])

In [72]:
def clean_text(text):
    text = text.lower()
    text = remove_urls(text)
    text = remove_stopwords(text)
    text = remove_emojis(text)
    text = handle_whitespace(text)
    text = lemmitization(text)
    text = stemminization(text)

    return text

In [74]:
# Convert comments column to str type
df['comments']=df['comments'].astype(str)

In [77]:
df['cleaned_comments']=df['comments'].apply(clean_text)
df.head()

,date,reviewer_name,comments,cleaned_comments
0,2014-07-01,Kristen,Large apartment; nice kitchen and bathroom. Ke...,larg apartment; nice kitchen bathroom. kenneth...
1,2014-10-24,Christopher,"This may be a little late, but just to say Ken...","may littl late, say kenneth quick respond requ..."
2,2015-03-04,Altay,The apartment was very clean and convenient to...,apart clean conveni downtown. one thing keep m...
3,2015-03-25,John,Kenneth was ready when I got there and arrange...,kenneth readi got arrang upstair neighbor meet...
4,2015-06-23,Jennifer,We were pleased to see how 2nd Street and the ...,pleas see 2nd street ten broeck neighborhood a...


In [81]:
# Remove any rows with null values
df=df.dropna()
df.isnull().sum()

date                0
reviewer_name       0
comments            0
cleaned_comments    0
dtype: int64

In [83]:
# Drop any rows which do not start with a number in the date column
df=df[df['date'].str.contains('^\d',na=False)]

In [85]:
df['date']=pd.to_datetime(df['date'])

In [144]:
# Store the Cleaned Data
df.to_csv('Dataset/Cleaned Data/Cleaned_reviews.csv', index=False)